In [5]:
# Multinomial Logistic Regression (MNL) Baseline Model for Travel Mode Choice

# This notebook implements a Multinomial Logistic Regression model as a baseline
# to predict individual travel mode choice. The model is trained using
# socio-demographic and trip-related variables and provides interpretable
# estimates of how these factors influence the probability of choosing each mode.

Imports

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, f1_score, classification_report

Load dataset

In [7]:
from pathlib import Path

filename = "Cleaned_Data.csv"
base_path = Path("C:/Users/youri") #Put your own base here

paths = list(base_path.rglob(filename))

for p in paths:
    print(p)

df = pd.read_csv(paths[0])

C:\Users\youri\OneDrive\Desktop\ME 44312 (Machine Learning)\Cleaned_Data.csv


Defining Baseline features

In [8]:
y = df["Choice"]

baseline_features = ["age", "Gender", "Income", "NbCar", "NbHousehold", "NbChild", "TimePT", "CostPT", "CostCar", "distance_km", "TripPurpose", "UrbRur", "Region"]

X = df[baseline_features]

Identify feature types:

In [9]:
categorical_cols = ["Gender", "TripPurpose", "UrbRur", "Region"]
numerical_cols = [col for col in X.columns if col not in categorical_cols]

Apply transformation to categorical features (Added a scalar for num vars since MNL requires it)

In [10]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

Define train/Test split

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test → 80% train
    random_state=42,    # reproducibility
    stratify=y          # preserve class distribution
)

Building the model:

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logit_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        multi_class="multinomial",
        max_iter=1000,
        class_weight="balanced"
    ))
])

Train the MNL

In [13]:
logit_model.fit(X_train, y_train)

c:\Users\youri\anaconda3\envs\TIL_Python\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Validate the MNL

In [14]:
y_pred_logit = logit_model.predict(X_test)

In [15]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred_logit))
print("F1 (macro):", f1_score(y_test, y_pred_logit, average="macro"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_logit))

Accuracy: 0.601123595505618
F1 (macro): 0.5048656723812355

Classification Report:
              precision    recall  f1-score   support

           0       0.51      0.57      0.54        44
           1       0.87      0.61      0.72       123
           2       0.16      0.64      0.26        11

    accuracy                           0.60       178
   macro avg       0.52      0.60      0.50       178
weighted avg       0.74      0.60      0.64       178



Extracting RUM's and betas

In [16]:
model = logit_model.named_steps["classifier"]
feature_names = logit_model.named_steps["preprocessor"].get_feature_names_out()

coef = model.coef_

In [17]:
import pandas as pd

coef_df = pd.DataFrame(coef, columns=feature_names)
coef_df.index = ["Public (0)", "Private (1)", "Soft (2)"]

coef_df

,num__age,num__Income,num__NbCar,num__NbHousehold,num__NbChild,num__TimePT,num__CostPT,num__CostCar,num__distance_km,cat__Gender_1,...,cat__UrbRur_1,cat__UrbRur_2,cat__Region_1,cat__Region_2,cat__Region_3,cat__Region_4,cat__Region_5,cat__Region_6,cat__Region_7,cat__Region_8
Public (0),-0.108701,-0.308907,-0.197974,0.370694,-0.257806,-0.334914,1.979866,0.894772,0.100359,0.063012,...,-0.084968,0.091369,0.283092,-0.034654,-0.508914,0.257514,0.010537,0.171079,0.330870,-0.503123
Private (1),0.199987,-0.001027,0.659245,-0.631945,0.570131,0.854475,1.209452,-0.063926,0.183831,-0.027185,...,0.127461,-0.121982,1.044135,0.238082,0.469357,-0.654667,-0.081891,-0.539177,-0.295619,-0.174741
Soft (2),-0.091286,0.309934,-0.461271,0.261251,-0.312326,-0.519562,-3.189318,-0.830847,-0.284190,-0.035827,...,-0.042494,0.030613,-1.327228,-0.203428,0.039558,0.397153,0.071354,0.368098,-0.035251,0.677865


In [19]:
def print_utility_full(coefs, feature_names, class_name):
    terms = []
    for coef, name in zip(coefs, feature_names):
        terms.append(f"{coef:.3f}·{name}")
    equation = " + ".join(terms)
    print(f"U_{class_name} = {equation}\n")


for i, class_name in enumerate(["Public", "Private", "Soft"]):
    print_utility_full(coef[i], feature_names, class_name)

U_Public = -0.109·num__age + -0.309·num__Income + -0.198·num__NbCar + 0.371·num__NbHousehold + -0.258·num__NbChild + -0.335·num__TimePT + 1.980·num__CostPT + 0.895·num__CostCar + 0.100·num__distance_km + 0.063·cat__Gender_1 + -0.057·cat__Gender_2 + 0.102·cat__TripPurpose_1 + 0.003·cat__TripPurpose_2 + -0.099·cat__TripPurpose_3 + -0.085·cat__UrbRur_1 + 0.091·cat__UrbRur_2 + 0.283·cat__Region_1 + -0.035·cat__Region_2 + -0.509·cat__Region_3 + 0.258·cat__Region_4 + 0.011·cat__Region_5 + 0.171·cat__Region_6 + 0.331·cat__Region_7 + -0.503·cat__Region_8

U_Private = 0.200·num__age + -0.001·num__Income + 0.659·num__NbCar + -0.632·num__NbHousehold + 0.570·num__NbChild + 0.854·num__TimePT + 1.209·num__CostPT + -0.064·num__CostCar + 0.184·num__distance_km + -0.027·cat__Gender_1 + 0.033·cat__Gender_2 + -0.218·cat__TripPurpose_1 + -0.009·cat__TripPurpose_2 + 0.232·cat__TripPurpose_3 + 0.127·cat__UrbRur_1 + -0.122·cat__UrbRur_2 + 1.044·cat__Region_1 + 0.238·cat__Region_2 + 0.469·cat__Region_3 + -0.6